In [22]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [23]:
result_path = Path("../results/downstream_task")
methods_list = ["uniform","psa", "kmm", "mrs-forest", 
               # "soft-mrs-exponential", 
                "fw-mrs-temperature_old",  
                "fw-mrs-temperature-svm_old",
                "fw-mrs-temperature",  "fw-mrs-temperature-svm",
                ]
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
metrics = ["AUROC", "AUPRC"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
datasets = ["folktables_employment", "folktables_income", "hr_analytics", "breast_cancer", "loan_prediction", "diabetes", "bank_marketing", 
            "german_credit"]
method_name_replacer = {"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                            # "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "mrs-forest": "MRS", "fw-mrs-temperature_old": "FW-MRS-RF-Abs", "fw-mrs-temperature-svm_old": "FW-MRS-SVM-Abs",
                        "fw-mrs-temperature-svm": "FW-MRS-SVM-Signed", "fw-mrs-temperature": "FW-MRS-RF-Signed",
                          }

In [24]:
aurocs = []
auprcs = []
dict_list = []
for dataset in datasets:
    for bias_type in bias_types:
        for method in methods_list:
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "classification_results.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "AUROC Mean": result_file["random forest auroc"]["mean"], 
                        "AUROC Std": result_file["random forest auroc"]["sd"], 
                        "AUPRC Mean": result_file["random forest auprc"]["mean"], 
                        "AUPRC Std": result_file["random forest auprc"]["sd"], 
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                        "Dropped Samples Mean": result_file["dropped_samples"]["mean"],
                        "Dropped Samples Std": result_file["dropped_samples"]["std"]
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [25]:
result_df = result_df.replace(method_name_replacer)
result_df

,Method,Data Set,AUROC Mean,AUROC Std,AUPRC Mean,AUPRC Std,Bias Type,Bias Strength,Dropped Samples Mean,Dropped Samples Std
0,Uniform,folktables_employment,0.871697,0.010310,0.827937,0.017155,less_positive_class,0.1,0.00,0.000000
1,PSA,folktables_employment,0.867177,0.011390,0.823004,0.017988,less_positive_class,0.1,0.00,0.000000
2,KMM,folktables_employment,0.859151,0.012266,0.812484,0.019343,less_positive_class,0.1,0.00,0.000000
3,MRS,folktables_employment,0.869787,0.010676,0.825169,0.017483,less_positive_class,0.1,300.60,122.006721
4,FW-MRS-RF-Signed,folktables_employment,0.865073,0.011916,0.820736,0.018674,less_positive_class,0.1,393.50,69.658094
5,FW-MRS-SVM-Signed,folktables_employment,0.834048,0.014466,0.776362,0.023446,less_positive_class,0.1,270.90,31.475228
6,Uniform,folktables_income,0.839249,0.011924,0.790149,0.016918,less_positive_class,0.1,0.00,0.000000
7,PSA,folktables_income,0.831454,0.013224,0.780936,0.018020,less_positive_class,0.1,0.06,0.310483
8,KMM,folktables_income,0.821094,0.014272,0.767031,0.019205,less_positive_class,0.1,0.00,0.000000
9,MRS,folktables_income,0.837711,0.012394,0.789246,0.018232,less_positive_class,0.1,307.50,87.676964


In [26]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auroc_values = []
            std_auroc_values = []
            for dataset in datasets:
                try:
                    mean_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Mean"].iloc[0]
                    mean_auroc_values.append(np.round(mean_auroc, 3))

                    std_auroc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUROC Std"].iloc[0]
                    std_auroc_values.append(np.round(std_auroc, 2))
                except IndexError:
                    mean_auroc_values.append(0)
                    std_auroc_values.append(0)

            print(f"\t& {method} \
& ${mean_auroc_values[0]}\\pm{std_auroc_values[0]}$ \
& ${mean_auroc_values[1]}\\pm{std_auroc_values[1]}$ \
& ${mean_auroc_values[2]}\\pm{std_auroc_values[2]}$ \
& ${mean_auroc_values[3]}\\pm{std_auroc_values[3]}$ \
& ${mean_auroc_values[4]}\\pm{std_auroc_values[4]}$ & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.872\pm0.01$ & $0.839\pm0.01$ & $0.753\pm0.02$ & $0.988\pm0.01$ & $0.669\pm0.07$ & \\
	& PSA & $0.867\pm0.01$ & $0.831\pm0.01$ & $0.751\pm0.02$ & $0.988\pm0.01$ & $0.623\pm0.09$ & \\
	& KMM & $0.859\pm0.01$ & $0.821\pm0.01$ & $0.749\pm0.02$ & $0.99\pm0.01$ & $0.607\pm0.08$ & \\
	& MRS & $0.87\pm0.01$ & $0.838\pm0.01$ & $0.751\pm0.02$ & $0.99\pm0.01$ & $0.652\pm0.08$ & \\
	& FW-MRS-RF-Signed & $0.865\pm0.01$ & $0.835\pm0.01$ & $0.75\pm0.02$ & $0.989\pm0.01$ & $0.607\pm0.09$ & \\
	& FW-MRS-SVM-Signed & $0.834\pm0.01$ & $0.828\pm0.01$ & $0.746\pm0.02$ & $0.985\pm0.01$ & $0.551\pm0.08$ & \\




In [27]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            mean_auprc_values = []
            std_auprc_values = []
            for dataset in datasets:
                try:
                    mean_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Mean"].iloc[0]
                    mean_auprc_values.append(np.round(mean_auprc, 3))

                    std_auprc = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                        (result_df["Bias Strength"]==bias_strength) & 
                                                        (result_df["Data Set"]==dataset)]["AUPRC Std"].iloc[0]
                    std_auprc_values.append(np.round(std_auprc, 2))
                except IndexError:
                    mean_auprc_values.append(0)
                    std_auprc_values.append(0)

            print(f"\t& {method} \
& ${mean_auprc_values[0]}\\pm{std_auprc_values[0]}$ \
& ${mean_auprc_values[1]}\\pm{std_auprc_values[1]}$ \
& ${mean_auprc_values[2]}\\pm{std_auprc_values[2]}$ \
& ${mean_auprc_values[3]}\\pm{std_auprc_values[3]}$ \
& ${mean_auprc_values[4]}\\pm{std_auprc_values[4]}$ & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & $0.828\pm0.02$ & $0.79\pm0.02$ & $0.459\pm0.02$ & $0.994\pm0.0$ & $0.802\pm0.05$ & \\
	& PSA & $0.823\pm0.02$ & $0.781\pm0.02$ & $0.456\pm0.03$ & $0.994\pm0.0$ & $0.783\pm0.06$ & \\
	& KMM & $0.812\pm0.02$ & $0.767\pm0.02$ & $0.448\pm0.03$ & $0.995\pm0.0$ & $0.774\pm0.05$ & \\
	& MRS & $0.825\pm0.02$ & $0.789\pm0.02$ & $0.457\pm0.03$ & $0.995\pm0.0$ & $0.796\pm0.05$ & \\
	& FW-MRS-RF-Signed & $0.821\pm0.02$ & $0.788\pm0.02$ & $0.453\pm0.03$ & $0.995\pm0.0$ & $0.771\pm0.05$ & \\
	& FW-MRS-SVM-Signed & $0.776\pm0.02$ & $0.78\pm0.02$ & $0.445\pm0.03$ & $0.993\pm0.0$ & $0.738\pm0.05$ & \\




In [28]:
result_df["Rank AUROC"] = result_df.groupby("Data Set")["AUROC Mean"].rank(ascending=False)
result_df["Rank AUPRC"] = result_df.groupby("Data Set")["AUPRC Mean"].rank(ascending=False)
result_df[["Method", "Rank AUROC", "Rank AUPRC"]].groupby("Method").mean()

,Rank AUROC,Rank AUPRC
Method,,
FW-MRS-RF-Signed,3.500,3.750
FW-MRS-SVM-Signed,5.500,5.375
KMM,4.875,4.875
MRS,2.000,1.750
PSA,3.625,3.625
Uniform,1.500,1.625
